# Interactive Coding Lab: Financial Time Series Analysis

This lab introduces a practical workflow for analyzing financial time series data across multiple asset classes using Python and `pandas`.

We will cover:
- loading and inspecting a dataset
- datetime handling and indexing
- resampling to weekly and monthly frequency
- simple and log returns
- cumulative performance visualization
- basic risk and reward analysis


## 1) Import Libraries

We use a minimal stack for this lab: `pandas` for time series analysis, `numpy` for numerical work, and `matplotlib` for visualization.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.options.display.float_format = '{:,.6f}'.format
plt.style.use('default')

## 2) Load and Inspect the Dataset

Before doing any time series analysis, it is important to inspect a new dataset.

This helps confirm:
- the file loaded correctly
- the column names are as expected
- the data types make sense
- the dataset is suitable for downstream calculations

In [6]:
candidate_paths = [
    Path('equity_fixed_income.csv'),
    Path('pandas-financial-data-analysis/equity_fixed_income.csv'),
    Path('close.csv'),
    Path('pandas-financial-data-analysis/close.csv'),
]

csv_path = next((path for path in candidate_paths if path.exists()), None)

if csv_path is None:
    searched_paths = '\n'.join(str(path.resolve()) for path in candidate_paths)
    raise FileNotFoundError(
        'Could not find a usable dataset. Looked for:\n'
        f'{searched_paths}'
    )

data = pd.read_csv(csv_path)
print(f'Loaded dataset: {csv_path}')
data.head()

Loaded dataset: close.csv


,Date,BA,BTC-USD,EURUSD=X,GC=F,MSFT,^DJI
0,2014-10-01,108.406662,383.614990,1.262834,"1,214.599976",38.880169,"16,804.710938"
1,2014-10-02,107.971878,375.071991,1.262419,"1,214.199951",38.761570,"16,801.050781"
2,2014-10-03,109.876221,359.511993,1.267058,"1,192.199951",39.041100,"17,009.689453"
3,2014-10-04,NaN,328.865997,NaN,NaN,NaN,NaN
4,2014-10-05,NaN,320.510010,NaN,NaN,NaN,NaN


In [7]:
data.info()
print('\nDtypes:')
print(data.dtypes)

if 'Date' not in data.columns:
    raise ValueError("The dataset must contain a 'Date' column.")

asset_columns = [column for column in data.columns if column != 'Date']

print(f"\nAsset-class columns detected: {asset_columns}")
print(f"Number of asset-class columns: {len(asset_columns)}")

if len(asset_columns) < 5:
    raise ValueError('Expected at least five asset-class columns in addition to Date.')

if len(asset_columns) > 5:
    selected_columns = asset_columns[:5]
    print(f"\nUsing the first five columns for the lab: {selected_columns}")
    data = data[['Date'] + selected_columns].copy()
    asset_columns = selected_columns
else:
    print('\nUsing all five asset-class columns in the dataset.')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2434 entries, 0 to 2433
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      2434 non-null   object 
 1   BA        1677 non-null   float64
 2   BTC-USD   2434 non-null   float64
 3   EURUSD=X  1735 non-null   float64
 4   GC=F      1674 non-null   float64
 5   MSFT      1677 non-null   float64
 6   ^DJI      1677 non-null   float64
dtypes: float64(6), object(1)
memory usage: 133.2+ KB

Dtypes:
Date         object
BA          float64
BTC-USD     float64
EURUSD=X    float64
GC=F        float64
MSFT        float64
^DJI        float64
dtype: object

Asset-class columns detected: ['BA', 'BTC-USD', 'EURUSD=X', 'GC=F', 'MSFT', '^DJI']
Number of asset-class columns: 6

Using the first five columns for the lab: ['BA', 'BTC-USD', 'EURUSD=X', 'GC=F', 'MSFT']


## 3) Create a DateTime Index

A datetime index is essential for time series work because it enables:
- chronological sorting
- date-based filtering
- resampling to different frequencies
- rolling and calendar-based analytics

In [ ]:
data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
data = data.dropna(subset=['Date']).set_index('Date').sort_index()

data.head()

In [ ]:
print(f"Start date: {data.index.min().date()}")
print(f"End date:   {data.index.max().date()}")

## 4) Resample to Weekly and Monthly Frequency

Downsampling reduces noise and makes broader trends easier to see.

Here we keep the **last available price** in each period:
- weekly using Friday as the week-end
- monthly using month-end

In [ ]:
weekly_data = data.resample('W-FRI').last().dropna(how='all')
monthly_data = data.resample('ME').last().dropna(how='all')

print('Weekly head:')
print(weekly_data.head())

print('\nMonthly head:')
print(monthly_data.head())

In [ ]:
row_comparison = pd.DataFrame({
    'Frequency': ['Daily', 'Weekly', 'Monthly'],
    'Rows': [len(data), len(weekly_data), len(monthly_data)]
})
row_comparison

## 5) Calculate Returns

We compute two common return definitions:
- **Simple returns** using `pct_change()`
- **Log returns** using `ln(P_t / P_(t-1))`

Simple returns are intuitive percentage changes. Log returns are convenient because they are additive over time.

In [ ]:
simple_daily_returns = data.pct_change(fill_method=None).dropna(how='all')
log_daily_returns = np.log(data / data.shift(1)).dropna(how='all')

print('Simple returns head:')
print(simple_daily_returns.head())

print('\nLog returns head:')
print(log_daily_returns.head())

## 6) Calculate Cumulative Returns

There are two standard ways to build cumulative growth series:
- from simple returns: `(1 + returns).cumprod()`
- from log returns: `exp(cumulative log returns)`

These should produce very similar cumulative growth paths.

In [ ]:
cumulative_simple = (1 + simple_daily_returns).cumprod()
cumulative_log = np.exp(log_daily_returns.cumsum())

print('Cumulative returns from simple returns:')
print(cumulative_simple.head())

print('\nCumulative returns from log returns:')
print(cumulative_log.head())

## 7) Visualize Asset Performance

Visual inspection often reveals patterns that summary statistics alone can miss.

We will look at:
- raw daily price levels
- cumulative growth series
- return distributions
- the smoothing effect of monthly downsampling

In [ ]:
plt.figure(figsize=(12, 6))
for column in data.columns:
    plt.plot(data.index, data[column], label=column)

plt.title('Daily Price Series by Asset Class')
plt.xlabel('Date')
plt.ylabel('Index Level / Price')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
for column in cumulative_simple.columns:
    plt.plot(cumulative_simple.index, cumulative_simple[column], label=column)

plt.title('Cumulative Returns by Asset Class')
plt.xlabel('Date')
plt.ylabel('Growth of 1')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 10))
axes = axes.flatten()

for axis, column in zip(axes, simple_daily_returns.columns):
    axis.hist(simple_daily_returns[column].dropna(), bins=30)
    axis.set_title(f'{column} Daily Simple Returns')
    axis.set_xlabel('Return')
    axis.set_ylabel('Frequency')

# Hide any unused subplot if needed.
for axis in axes[len(simple_daily_returns.columns):]:
    axis.set_visible(False)

figure.tight_layout()
plt.show()

In [ ]:
selected_asset = data.columns[0]

plt.figure(figsize=(12, 6))
plt.plot(data.index, data[selected_asset], label=f'{selected_asset} Daily')
plt.plot(monthly_data.index, monthly_data[selected_asset], label=f'{selected_asset} Monthly', linewidth=2)
plt.title(f'Daily vs Monthly Prices: {selected_asset}')
plt.xlabel('Date')
plt.ylabel('Price / Index Level')
plt.legend()
plt.tight_layout()
plt.show()

## 8) Analyze Risk and Reward

A simple first-pass comparison uses:
- average daily simple return as a basic reward measure
- standard deviation of daily simple returns as a basic volatility measure

In [ ]:
average_daily_return = simple_daily_returns.mean()
daily_volatility = simple_daily_returns.std()

risk_return_summary = pd.DataFrame({
    'Average Daily Return': average_daily_return,
    'Daily Volatility': daily_volatility
}).sort_values(by='Average Daily Return', ascending=False)

risk_return_summary

### Interpreting the summary table

Questions to ask:
- Which asset classes had the highest average daily return?
- Which had the highest volatility?
- Were equity-like series generally more volatile than fixed income?
- Do the summary metrics match what the charts suggested?

This is a useful check: the visual story and the numerical summary should broadly agree.

## 9) Final Conclusions

This lab showed how to move from raw financial time series to a first-pass multi-asset analysis.

Key ideas:
- different asset classes can show very different performance paths
- resampling smooths noisy time series and highlights broader trends
- average return and volatility provide a simple framework for comparing risk and reward
- combining charts with summary statistics gives a more balanced view than either alone

## Further Exploration

Three natural next steps:
- annualize returns and volatility
- calculate rolling volatility
- compare correlations between asset classes